# Volume 2, Appendix A: Spaces, Quotients and Duality (Python companion)

Exercises first, solutions at the end. We compute dual bases and use them for interpolation and quadrature, solve linear systems over $\mathbb F_2$ with a certificate when there is no solution (the Fredholm alternative), play Lights Out on boards of every size, test the dimension argument on random club systems, and find unisolvent points. Only `numpy` and `matplotlib` are needed.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
rng = np.random.default_rng(1); random.seed(1)

## Exercises

**P1.1 (★) Dual bases.** For a basis $e_1,\dots,e_n$ of $\mathbb R^n$ given as the columns of an invertible matrix $P$, show that the dual basis $e_1^*,\dots,e_n^*$ is given by the rows of $P^{-1}$, and check it on a random $5\times5$ matrix. Then check Proposition 1.7 (each form $\varphi$ equals $\sum_i\varphi(e_i)e_i^*$) on a random row vector $\varphi$.

**P1.2 (★) Lagrange interpolation and its limits.** Implement the Lagrange basis of Example 1.9 and interpolate $f(x)=1/(1+25x^2)$ on $[-1,1]$ at $n+1$ equally spaced points for $n=5,10,15$. Plot the interpolants. What happens near the endpoints (this is Runge's phenomenon)? Repeat with the Chebyshev points $\cos\frac{(2j+1)\pi}{2n+2}$.

**P1.3 (★★) Quadrature weights from the dual basis.** Exercise 1.7 says that integration on $[0,1]$ is $\sum_iw_i\varepsilon_i$ with $w_i=\int_0^1L_i$. Compute the weights exactly (with fractions) for equally spaced points $0,\frac1n,\dots,1$ and $n=1,\dots,10$. Recover the trapezoid and Simpson rules. For which $n$ does a negative weight appear, and why is that bad news numerically?

**P1.4 (★★) Gaussian elimination over $\mathbb F_2$ with a certificate.** Write `solve_f2(A, b)` that returns either a solution of $Ax=b$ over $\mathbb F_2$, or a vector $y$ with $A^{\mathsf T}y=0$ and $y\cdot b=1$ (Corollary 1.14). Test it on $500$ random systems, checking every answer.

**P1.5 (★★) Lights Out on every board.** Build the $n^2\times n^2$ Lights Out matrix of Example 1.15 and compute the dimension of its kernel over $\mathbb F_2$ for $n=1,\dots,30$. On which boards is every configuration solvable? For $n=5$, find the two quiet patterns, draw a random solvable configuration, solve it, and display the solution.

**P1.6 (★) Oddtown and Reverse Oddtown.** (a) For $n=6$, build $6$ odd clubs with even pairwise intersections other than the singletons. (b) For small $n$, search for the largest family of *even* clubs with pairwise *odd* intersections by a randomised greedy search, and compare with the bound $n+1$ of Exercise 1.10.

**P1.7 (★) An invertible matrix in every hyperplane.** Follow the proof of Theorem 1.18: given a random non-zero $A\in M_n(\mathbb R)$ (of any rank), find invertible $P,Q$ with $A=PJ_rQ$ and build an invertible $X$ with $\operatorname{tr}(AX)=0$. Check it for $n=2,\dots,8$ and ranks $1,\dots,n$.

**P1.8 (★★) Unisolvent points in the plane.** The polynomials of degree at most $2$ in $x,y$ form a space of dimension $6$. Check that $6$ random points are unisolvent (the $6\times6$ evaluation matrix is invertible), while $6$ points on a circle never are; compute the rank in that case and explain it. What about $6$ points on a parabola?

## Solutions

### P1.1
**How to find it.** $e_i^*(e_j)=\delta_{ij}$ says (row $i$ of $Q$) times (column $j$ of $P$) is $\delta_{ij}$, that is, $QP=I$.

In [ ]:
n = 5
P = rng.normal(size=(n, n))
Q = np.linalg.inv(P)                 # rows = dual basis
print("max |e_i^*(e_j) - delta_ij| =", np.abs(Q @ P - np.eye(n)).max())
phi = rng.normal(size=n)             # a linear form, as a row vector
recon = sum((phi @ P[:, i]) * Q[i] for i in range(n))
print("max |phi - sum phi(e_i) e_i^*| =", np.abs(phi - recon).max())

### P1.2
**How to find it.** $L_j(x)=\prod_{k\neq j}\frac{x-x_k}{x_j-x_k}$; the interpolant is $\sum_j f(x_j)L_j$. The Lagrange polynomials of equally spaced points grow enormously near the ends, and any small error is amplified.

In [ ]:
def lagrange_eval(xs, ys, t):
    t = np.asarray(t, dtype=float); out = np.zeros_like(t)
    for j, xj in enumerate(xs):
        Lj = np.ones_like(t)
        for k, xk in enumerate(xs):
            if k != j: Lj *= (t - xk) / (xj - xk)
        out += ys[j] * Lj
    return out

f = lambda x: 1 / (1 + 25 * x**2)
t = np.linspace(-1, 1, 1001)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, kind in zip(axes, ["equally spaced", "Chebyshev"]):
    ax.plot(t, f(t), 'k', lw=2, label='f')
    for n in (5, 10, 15):
        xs = np.linspace(-1, 1, n + 1) if kind == "equally spaced" else np.cos((2*np.arange(n + 1) + 1) * np.pi / (2*n + 2))
        ax.plot(t, lagrange_eval(xs, f(xs), t), label=f'n={n}')
    ax.set_ylim(-1, 2); ax.set_title(kind + " points"); ax.legend()
plt.show()
for n in (5, 10, 15, 20):
    xs = np.linspace(-1, 1, n + 1)
    print(n, "max error, equally spaced: %.3g" % np.abs(lagrange_eval(xs, f(xs), t) - f(t)).max(),
          " Chebyshev: %.3g" % np.abs(lagrange_eval(np.cos((2*np.arange(n+1)+1)*np.pi/(2*n+2)), f(np.cos((2*np.arange(n+1)+1)*np.pi/(2*n+2))), t) - f(t)).max())

With equally spaced points the error *grows* with $n$ near the endpoints; with Chebyshev points it decreases. Interpolation is linear algebra, but the basis matters: Chapter 4 explains Chebyshev points through orthogonal polynomials.

### P1.3
**How to find it.** Expand $L_i$ as a polynomial with rational coefficients and integrate term by term: $\int_0^1t^k\,dt=\frac1{k+1}$.

In [ ]:
def poly_mul(p, q):
    r = [Fraction(0)] * (len(p) + len(q) - 1)
    for i, a in enumerate(p):
        for j, b in enumerate(q):
            r[i + j] += a * b
    return r

def newton_cotes(n):
    xs = [Fraction(i, n) for i in range(n + 1)]
    ws = []
    for i in range(n + 1):
        p = [Fraction(1)]
        for k in range(n + 1):
            if k != i:
                p = poly_mul(p, [-xs[k] / (xs[i] - xs[k]), 1 / (xs[i] - xs[k])])
        ws.append(sum(c / (d + 1) for d, c in enumerate(p)))
    return ws

for n in range(1, 11):
    w = newton_cotes(n)
    print(n, [str(x) for x in w], " negative weight!" if min(w) < 0 else "")

$n=1$ is the trapezoid rule $(\frac12,\frac12)$, $n=2$ is Simpson $(\frac16,\frac23,\frac16)$. The first negative weight appears at $n=8$; then $\sum|w_i|>1$, so rounding errors in the values $f(x_i)$ are amplified, and the rules become useless for large $n$. Gauss's choice of points (Chapter 4) keeps all weights positive.

### P1.4
**How to find it.** Row-reduce the augmented matrix $[A\mid b]$ while recording the row operations in a matrix $E$ (so that $EA$ is in echelon form). If a row of $EA$ is zero but the corresponding entry of $Eb$ is $1$, that row of $E$ is the certificate $y$: $y^{\mathsf T}A=0$ and $y^{\mathsf T}b=1$.

In [ ]:
def solve_f2(A, b):
    A = np.array(A, dtype=np.uint8) % 2; b = np.array(b, dtype=np.uint8) % 2
    m, n = A.shape
    M = np.concatenate([A, b[:, None], np.eye(m, dtype=np.uint8)], axis=1)   # [A | b | E]
    r, pivots = 0, []
    for c in range(n):
        rows = [i for i in range(r, m) if M[i, c]]
        if not rows: continue
        M[[r, rows[0]]] = M[[rows[0], r]]
        for i in range(m):
            if i != r and M[i, c]: M[i] ^= M[r]
        pivots.append(c); r += 1
    for i in range(r, m):                     # zero rows of the reduced A
        if M[i, n]:
            return ('certificate', M[i, n + 1:].copy())
    x = np.zeros(n, dtype=np.uint8)
    for i, c in enumerate(pivots):
        x[c] = M[i, n]
    return ('solution', x)

counts = {'solution': 0, 'certificate': 0}
for _ in range(500):
    m, n = rng.integers(2, 12, size=2)
    A = rng.integers(0, 2, size=(m, n)); b = rng.integers(0, 2, size=m)
    kind, v = solve_f2(A, b)
    if kind == 'solution':
        assert ((A @ v - b) % 2 == 0).all()
    else:
        assert ((v @ A) % 2 == 0).all() and (v @ b) % 2 == 1
    counts[kind] += 1
print(counts, "- every answer checked")

### P1.5
**How to find it.** The kernel dimension is $n^2-\operatorname{rank}_{\mathbb F_2}A$; the rank comes from the same elimination. Quiet patterns are kernel vectors, read off from the free columns.

In [ ]:
def lights_matrix(n):
    N = n * n; A = np.zeros((N, N), dtype=np.uint8)
    for i in range(n):
        for j in range(n):
            for di, dj in ((0, 0), (1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < n and 0 <= b < n:
                    A[i * n + j, a * n + b] = 1
    return A

def kernel_f2(A):
    M = A.copy() % 2; m, n = M.shape; r, piv = 0, []
    for c in range(n):
        rows = np.nonzero(M[r:, c])[0]
        if len(rows) == 0: continue
        p = r + rows[0]; M[[r, p]] = M[[p, r]]
        mask = M[:, c].astype(bool); mask[r] = False
        M[mask] ^= M[r]
        piv.append(c); r += 1
        if r == m: break
    free = [c for c in range(n) if c not in set(piv)]
    basis = []
    for f in free:
        x = np.zeros(n, dtype=np.uint8); x[f] = 1
        for i, c in enumerate(piv): x[c] = M[i, f]
        basis.append(x)
    return basis

dims = {n: len(kernel_f2(lights_matrix(n))) for n in range(1, 31)}
print("kernel dimension by board size:", dims)
print("boards where everything is solvable:", [n for n, d in dims.items() if d == 0])

In [ ]:
A = lights_matrix(5)
quiet = kernel_f2(A)
for q in quiet:
    assert ((A @ q) % 2 == 0).all()
x0 = rng.integers(0, 2, size=25).astype(np.uint8)
b = (A @ x0) % 2                     # a solvable configuration by construction
print("parity checks q.b:", [int(q @ b % 2) for q in quiet])
kind, x = solve_f2(A, b)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, M, title in zip(axes, [quiet[0], quiet[1], b, x], ["quiet pattern 1", "quiet pattern 2", "configuration", "buttons to press"]):
    ax.imshow(M.reshape(5, 5), cmap='YlOrBr', vmin=0, vmax=1.3); ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
plt.show()
print("solution checks:", ((A @ x) % 2 == b).all())

### P1.6
**How to find it.** (a) Odd sizes, even intersections: take disjoint odd blocks, or the complements of single points when $n$ is even ($n-1$ is odd and two complements meet in $n-2$ points, which is even). (b) Greedy: keep adding a random even set compatible with all chosen ones.

In [ ]:
n = 6
clubs = [set(range(n)) - {i} for i in range(n)]
print("sizes:", [len(c) for c in clubs], " intersections:", sorted({len(a & b) for a, b in itertools.combinations(clubs, 2)}))

def reverse_oddtown_search(n, tries=4000):
    universe = [frozenset(s) for k in range(2, n + 1, 2) for s in itertools.combinations(range(n), k)]
    best = []
    for _ in range(tries):
        random.shuffle(universe); fam = []
        for s in universe:
            if all(len(s & t) % 2 == 1 for t in fam):
                fam.append(s)
        if len(fam) > len(best): best = fam
    return best

for n in range(2, 9):
    fam = reverse_oddtown_search(n, 300 if n > 6 else 2000)
    print(f"n={n}: found {len(fam)} even clubs with odd intersections (bound n+1 = {n+1}; n if the count is even)")

The search finds $n$ for odd $n$ and $n-1$ for even $n$. The exact answer is known (Babai and Frankl, Chapter 1): our bound $n+1$ is never attained, and a finer argument with the all-ones vector gives the truth.

### P1.7
**How to find it.** The singular value decomposition gives $A=U\Sigma V^{\mathsf T}$, and scaling the non-zero singular values to $1$ gives $A=PJ_rQ$ with $P=U\,\mathrm{diag}(\sigma_1,\dots,\sigma_r,1,\dots,1)$ and $Q=V^{\mathsf T}$. Then $X=Q^{-1}CP^{-1}$ with $C$ a cyclic permutation matrix.

In [ ]:
def invertible_in_hyperplane(A):
    n = A.shape[0]
    U, s, Vt = np.linalg.svd(A)
    r = int((s > 1e-10 * s[0]).sum())
    d = np.ones(n); d[:r] = s[:r]
    P, Q = U @ np.diag(d), Vt
    Cyc = np.roll(np.eye(n), 1, axis=0)          # e_i -> e_{i+1}
    return np.linalg.inv(Q) @ Cyc @ np.linalg.inv(P)

worst = 0
for n in range(2, 9):
    for r in range(1, n + 1):
        A = rng.normal(size=(n, r)) @ rng.normal(size=(r, n))
        X = invertible_in_hyperplane(A)
        worst = max(worst, abs(np.trace(A @ X)))
        assert abs(np.linalg.det(X)) > 1e-8
print("largest |tr(AX)| over all tests:", worst)

### P1.8
**How to find it.** The evaluation matrix has rows $(1,x,y,x^2,xy,y^2)$ at the six points; it is invertible exactly when the points are unisolvent. On the circle $x^2+y^2=1$ the polynomial $x^2+y^2-1$ vanishes at every point, so the columns satisfy a linear relation and the rank is at most $5$.

In [ ]:
def evals(pts):
    return np.array([[1, x, y, x*x, x*y, y*y] for x, y in pts])

rand_pts = rng.normal(size=(6, 2))
theta = rng.uniform(0, 2*np.pi, 6); circle = np.c_[np.cos(theta), np.sin(theta)]
u = rng.normal(size=6); parabola = np.c_[u, u**2]
for name, pts in [("random", rand_pts), ("circle", circle), ("parabola", parabola)]:
    print(f"{name:9s} rank = {np.linalg.matrix_rank(evals(pts))}")

Six points on any conic fail for the same reason: the equation of the conic is a non-zero polynomial of degree $2$ vanishing at all of them. Six random points lie on no common conic (five points determine a conic: Volume 8, Appendix E), which is why they are unisolvent.